In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName("Banking Transactions Analytics").getOrCreate()
sc = spark.sparkContext

In [4]:
customers_csv = """customer_id,name,city,age,account_type,signup_date
1,Rahul,Hyderabad,29,Savings,2023-01-10
2,Sneha,Bangalore,32,Current,2023-02-12
3,Arjun,Mumbai,27,Savings,2023-03-14
4,Priya,Delhi,35,Savings,2023-04-15
5,Karan,Chennai,30,Current,2023-05-11
6,Meera,Hyderabad,31,Savings,2023-06-10
7,Amit,Pune,38,Current,2023-06-22
8,Neha,Delhi,26,Savings,2023-07-10
9,Divya,Bangalore,28,Savings,2023-07-15
10,Vikram,Mumbai,42,Current,2023-08-01
11,Farhan,Hyderabad,34,Savings,2023-08-10
12,Simran,Delhi,25,Savings,2023-08-21
"""

accounts_csv = """account_id,customer_id,branch,balance
1001,1,Hyderabad Main,85000
1002,2,Bangalore Central,120000
1003,3,Mumbai West,45000
1004,4,Delhi North,95000
1005,5,Chennai South,60000
1006,6,Hyderabad Main,150000
1007,7,Pune East,30000
1008,8,Delhi North,70000
1009,9,Bangalore Central,110000
1010,10,Mumbai West,200000
1011,11,Hyderabad Main,50000
1012,12,Delhi North,40000
"""

transactions_csv = """txn_id,account_id,txn_type,amount,txn_date
1,1001,Credit,25000,2024-03-01
2,1002,Debit,15000,2024-03-01
3,1003,Credit,10000,2024-03-02
4,1004,Debit,5000,2024-03-02
5,1005,Credit,30000,2024-03-03
6,1006,Debit,20000,2024-03-03
7,1007,Credit,8000,2024-03-04
8,1008,Debit,12000,2024-03-04
9,1009,Credit,40000,2024-03-05
10,1010,Debit,35000,2024-03-05
11,1001,Debit,7000,2024-03-06
12,1002,Credit,18000,2024-03-06
13,1006,Credit,50000,2024-03-07
14,1010,Credit,60000,2024-03-07
15,1011,Debit,9000,2024-03-08
16,1012,Credit,16000,2024-03-08
17,1003,Debit,4000,2024-03-09
18,1004,Credit,22000,2024-03-09
19,1005,Debit,11000,2024-03-10
20,1009,Debit,14000,2024-03-10
"""

branches_csv = """branch,region,manager
Hyderabad Main,South,Ramesh
Bangalore Central,South,Leena
Mumbai West,West,Joseph
Delhi North,North,Sara
Chennai South,South,Kumar
Pune East,West,Anita
"""

logs_txt = """Rahul login
Sneha login
Rahul transfer
Arjun login
Priya withdrawal
Rahul logout
Meera login
Vikram transfer
Divya login
Farhan login
Simran withdrawal
Neha login
Amit deposit
Karan login
Meera transfer
Vikram login
Rahul deposit
Sneha withdrawal
Farhan transfer
Divya logout
"""

customer_profiles_json = """[
  {
    "customer_id": 1,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000011111"},
    "services": ["UPI", "Credit Card", "Net Banking"]
  },
  {
    "customer_id": 2,
    "name": "Sneha",
    "contact": {"email": "sneha@mail.com", "phone": "9000022222"},
    "services": ["UPI", "Debit Card"]
  },
  {
    "customer_id": 3,
    "name": "Arjun",
    "contact": {"email": "arjun@mail.com", "phone": "9000033333"},
    "services": ["Net Banking", "Loan"]
  },
  {
    "customer_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "services": ["UPI", "Credit Card", "Loan"]
  },
  {
    "customer_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "services": ["Net Banking", "Wealth"]
  }
]
"""

In [5]:
with open("bank_customers.csv", "w") as f:
    f.write(customers_csv)

with open("accounts.csv", "w") as f:
    f.write(accounts_csv)

with open("transactions.csv", "w") as f:
    f.write(transactions_csv)

with open("branches.csv", "w") as f:
    f.write(branches_csv)

with open("bank_logs.txt", "w") as f:
    f.write(logs_txt)

with open("customer_profiles.json", "w") as f:
    f.write(customer_profiles_json)

print("Banking datasets created")

Banking datasets created


In [6]:
customers = spark.read.csv("bank_customers.csv", header=True, inferSchema=True)
accounts = spark.read.csv("accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
branches = spark.read.csv("branches.csv", header=True, inferSchema=True)
profiles = spark.read.json("customer_profiles.json", multiLine=True)

**Section 1 — DataFrame Basics**

In [7]:
#1
customers.show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [8]:
#2
accounts.show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1001|          1|   Hyderabad Main|  85000|
|      1002|          2|Bangalore Central| 120000|
|      1003|          3|      Mumbai West|  45000|
|      1004|          4|      Delhi North|  95000|
|      1005|          5|    Chennai South|  60000|
|      1006|          6|   Hyderabad Main| 150000|
|      1007|          7|        Pune East|  30000|
|      1008|          8|      Delhi North|  70000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
|      1011|         11|   Hyderabad Main|  50000|
|      1012|         12|      Delhi North|  40000|
+----------+-----------+-----------------+-------+



In [9]:
#3
transactions.show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [10]:
#4
branches.show()

+-----------------+------+-------+
|           branch|region|manager|
+-----------------+------+-------+
|   Hyderabad Main| South| Ramesh|
|Bangalore Central| South|  Leena|
|      Mumbai West|  West| Joseph|
|      Delhi North| North|   Sara|
|    Chennai South| South|  Kumar|
|        Pune East|  West|  Anita|
+-----------------+------+-------+



In [11]:
#5
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [12]:
#6
transactions.printSchema()

root
 |-- txn_id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- txn_date: date (nullable = true)



In [13]:
#7
customers.count()

12

In [14]:
#8
accounts.count()

12

In [15]:
#9
transactions.count()

20

In [16]:
#10
transactions.show(5)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
+------+----------+--------+------+----------+
only showing top 5 rows


**Section 2 — Select, Rename, Filter**

In [17]:
#11
customers.select("name", "city", "account_type").show()

+------+---------+------------+
|  name|     city|account_type|
+------+---------+------------+
| Rahul|Hyderabad|     Savings|
| Sneha|Bangalore|     Current|
| Arjun|   Mumbai|     Savings|
| Priya|    Delhi|     Savings|
| Karan|  Chennai|     Current|
| Meera|Hyderabad|     Savings|
|  Amit|     Pune|     Current|
|  Neha|    Delhi|     Savings|
| Divya|Bangalore|     Savings|
|Vikram|   Mumbai|     Current|
|Farhan|Hyderabad|     Savings|
|Simran|    Delhi|     Savings|
+------+---------+------------+



In [18]:
#12
accounts.select("account_id", "balance").show()

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|  85000|
|      1002| 120000|
|      1003|  45000|
|      1004|  95000|
|      1005|  60000|
|      1006| 150000|
|      1007|  30000|
|      1008|  70000|
|      1009| 110000|
|      1010| 200000|
|      1011|  50000|
|      1012|  40000|
+----------+-------+



In [19]:
#13
accounts.withColumnRenamed("balance", "current_balance").show()

+----------+-----------+-----------------+---------------+
|account_id|customer_id|           branch|current_balance|
+----------+-----------+-----------------+---------------+
|      1001|          1|   Hyderabad Main|          85000|
|      1002|          2|Bangalore Central|         120000|
|      1003|          3|      Mumbai West|          45000|
|      1004|          4|      Delhi North|          95000|
|      1005|          5|    Chennai South|          60000|
|      1006|          6|   Hyderabad Main|         150000|
|      1007|          7|        Pune East|          30000|
|      1008|          8|      Delhi North|          70000|
|      1009|          9|Bangalore Central|         110000|
|      1010|         10|      Mumbai West|         200000|
|      1011|         11|   Hyderabad Main|          50000|
|      1012|         12|      Delhi North|          40000|
+----------+-----------+-----------------+---------------+



In [20]:
#14
transactions.withColumnRenamed("txn_type", "transaction_type").show()

+------+----------+----------------+------+----------+
|txn_id|account_id|transaction_type|amount|  txn_date|
+------+----------+----------------+------+----------+
|     1|      1001|          Credit| 25000|2024-03-01|
|     2|      1002|           Debit| 15000|2024-03-01|
|     3|      1003|          Credit| 10000|2024-03-02|
|     4|      1004|           Debit|  5000|2024-03-02|
|     5|      1005|          Credit| 30000|2024-03-03|
|     6|      1006|           Debit| 20000|2024-03-03|
|     7|      1007|          Credit|  8000|2024-03-04|
|     8|      1008|           Debit| 12000|2024-03-04|
|     9|      1009|          Credit| 40000|2024-03-05|
|    10|      1010|           Debit| 35000|2024-03-05|
|    11|      1001|           Debit|  7000|2024-03-06|
|    12|      1002|          Credit| 18000|2024-03-06|
|    13|      1006|          Credit| 50000|2024-03-07|
|    14|      1010|          Credit| 60000|2024-03-07|
|    15|      1011|           Debit|  9000|2024-03-08|
|    16|  

In [21]:
#15
customers.filter(col("city") == "Hyderabad").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [22]:
#16
customers.filter(col("age") > 30).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [23]:
#17
customers.filter(col("account_type") == "Savings").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [24]:
#18
accounts.filter(col("balance") > 100000).show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1002|          2|Bangalore Central| 120000|
|      1006|          6|   Hyderabad Main| 150000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
+----------+-----------+-----------------+-------+



In [25]:
#19
transactions.filter(col("txn_type") == "Credit").show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    16|      1012|  Credit| 16000|2024-03-08|
|    18|      1004|  Credit| 22000|2024-03-09|
+------+----------+--------+------+----------+



In [26]:
#20
transactions.filter(col("amount") > 20000).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     5|      1005|  Credit| 30000|2024-03-03|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    18|      1004|  Credit| 22000|2024-03-09|
+------+----------+--------+------+----------+



**Section 3 — Sorting and Limit**

In [27]:
#21
customers.orderBy("age").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
+-----------+------+---------+---+------------+-----------+



In [28]:
#22
customers.orderBy(col("age").desc()).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [29]:
#23
accounts.orderBy(col("balance").desc()).show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1010|         10|      Mumbai West| 200000|
|      1006|          6|   Hyderabad Main| 150000|
|      1002|          2|Bangalore Central| 120000|
|      1009|          9|Bangalore Central| 110000|
|      1004|          4|      Delhi North|  95000|
|      1001|          1|   Hyderabad Main|  85000|
|      1008|          8|      Delhi North|  70000|
|      1005|          5|    Chennai South|  60000|
|      1011|         11|   Hyderabad Main|  50000|
|      1003|          3|      Mumbai West|  45000|
|      1012|         12|      Delhi North|  40000|
|      1007|          7|        Pune East|  30000|
+----------+-----------+-----------------+-------+



In [30]:
#24
accounts.orderBy(col("balance").desc()).show(5)

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1010|         10|      Mumbai West| 200000|
|      1006|          6|   Hyderabad Main| 150000|
|      1002|          2|Bangalore Central| 120000|
|      1009|          9|Bangalore Central| 110000|
|      1004|          4|      Delhi North|  95000|
+----------+-----------+-----------------+-------+
only showing top 5 rows


In [31]:
#25
accounts.orderBy("balance").show(5)

+----------+-----------+--------------+-------+
|account_id|customer_id|        branch|balance|
+----------+-----------+--------------+-------+
|      1007|          7|     Pune East|  30000|
|      1012|         12|   Delhi North|  40000|
|      1003|          3|   Mumbai West|  45000|
|      1011|         11|Hyderabad Main|  50000|
|      1005|          5| Chennai South|  60000|
+----------+-----------+--------------+-------+
only showing top 5 rows


In [32]:
#26
transactions.orderBy(col("amount").desc()).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|    14|      1010|  Credit| 60000|2024-03-07|
|    13|      1006|  Credit| 50000|2024-03-07|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|     5|      1005|  Credit| 30000|2024-03-03|
|     1|      1001|  Credit| 25000|2024-03-01|
|    18|      1004|  Credit| 22000|2024-03-09|
|     6|      1006|   Debit| 20000|2024-03-03|
|    12|      1002|  Credit| 18000|2024-03-06|
|    16|      1012|  Credit| 16000|2024-03-08|
|     2|      1002|   Debit| 15000|2024-03-01|
|    20|      1009|   Debit| 14000|2024-03-10|
|     8|      1008|   Debit| 12000|2024-03-04|
|    19|      1005|   Debit| 11000|2024-03-10|
|     3|      1003|  Credit| 10000|2024-03-02|
|    15|      1011|   Debit|  9000|2024-03-08|
|     7|      1007|  Credit|  8000|2024-03-04|
|    11|      1001|   Debit|  7000|2024-03-06|
|     4|     

In [33]:
#27
transactions.orderBy(col("amount").desc()).show(3)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|    14|      1010|  Credit| 60000|2024-03-07|
|    13|      1006|  Credit| 50000|2024-03-07|
|     9|      1009|  Credit| 40000|2024-03-05|
+------+----------+--------+------+----------+
only showing top 3 rows


In [34]:
#28
transactions.orderBy("txn_date").show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [35]:
#29
customers.orderBy("city", "age").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
+-----------+------+---------+---+------------+-----------+



In [36]:
#30
transactions.orderBy(col("txn_date").desc()).show(5)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|    19|      1005|   Debit| 11000|2024-03-10|
|    20|      1009|   Debit| 14000|2024-03-10|
|    18|      1004|  Credit| 22000|2024-03-09|
|    17|      1003|   Debit|  4000|2024-03-09|
|    15|      1011|   Debit|  9000|2024-03-08|
+------+----------+--------+------+----------+
only showing top 5 rows


**Section 4 — Aggregations**

In [37]:
#31
accounts.select(sum("balance")).show()

+------------+
|sum(balance)|
+------------+
|     1055000|
+------------+



In [38]:
#32
accounts.select(avg("balance")).show()

+-----------------+
|     avg(balance)|
+-----------------+
|87916.66666666667|
+-----------------+



In [39]:
#33
accounts.select(max("balance")).show()

+------------+
|max(balance)|
+------------+
|      200000|
+------------+



In [40]:
#34
accounts.select(min("balance")).show()

+------------+
|min(balance)|
+------------+
|       30000|
+------------+



In [41]:
#35
customers.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [42]:
#36
customers.groupBy("account_type").count().show()

+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|    8|
|     Current|    4|
+------------+-----+



In [43]:
#37
transactions.groupBy("txn_type").count().show()

+--------+-----+
|txn_type|count|
+--------+-----+
|  Credit|   10|
|   Debit|   10|
+--------+-----+



In [44]:
#38
transactions.filter(col("txn_type") == "Credit") \
            .select(sum("amount")) \
            .show()

+-----------+
|sum(amount)|
+-----------+
|     279000|
+-----------+



In [45]:
#39
transactions.filter(col("txn_type") == "Debit") \
            .select(sum("amount")) \
            .show()

+-----------+
|sum(amount)|
+-----------+
|     132000|
+-----------+



In [46]:
#40
transactions.select(avg("amount")).show()

+-----------+
|avg(amount)|
+-----------+
|    20550.0|
+-----------+



**Section 5 — GroupBy Analytics**

In [47]:
#41
accounts.groupBy("branch").agg(sum("balance")).show()

+-----------------+------------+
|           branch|sum(balance)|
+-----------------+------------+
|      Delhi North|      205000|
|   Hyderabad Main|      285000|
|        Pune East|       30000|
|      Mumbai West|      245000|
|    Chennai South|       60000|
|Bangalore Central|      230000|
+-----------------+------------+



In [48]:
#42
accounts.groupBy("branch").agg(avg("balance")).show()

+-----------------+-----------------+
|           branch|     avg(balance)|
+-----------------+-----------------+
|      Delhi North|68333.33333333333|
|   Hyderabad Main|          95000.0|
|        Pune East|          30000.0|
|      Mumbai West|         122500.0|
|    Chennai South|          60000.0|
|Bangalore Central|         115000.0|
+-----------------+-----------------+



In [49]:
#43
accounts.groupBy("branch").count().show()

+-----------------+-----+
|           branch|count|
+-----------------+-----+
|      Delhi North|    3|
|   Hyderabad Main|    3|
|        Pune East|    1|
|      Mumbai West|    2|
|    Chennai South|    1|
|Bangalore Central|    2|
+-----------------+-----+



In [50]:
#44
transactions.groupBy("account_id").agg(sum("amount")).show()

+----------+-----------+
|account_id|sum(amount)|
+----------+-----------+
|      1005|      41000|
|      1008|      12000|
|      1010|      95000|
|      1002|      33000|
|      1001|      32000|
|      1006|      70000|
|      1007|       8000|
|      1003|      14000|
|      1004|      27000|
|      1011|       9000|
|      1012|      16000|
|      1009|      54000|
+----------+-----------+



In [51]:
#45
transactions.groupBy("account_id").count().show()

+----------+-----+
|account_id|count|
+----------+-----+
|      1005|    2|
|      1008|    1|
|      1010|    2|
|      1002|    2|
|      1001|    2|
|      1006|    2|
|      1007|    1|
|      1003|    2|
|      1004|    2|
|      1011|    1|
|      1012|    1|
|      1009|    2|
+----------+-----+



In [52]:
#46
transactions.groupBy("txn_type").agg(sum("amount")).show()

+--------+-----------+
|txn_type|sum(amount)|
+--------+-----------+
|  Credit|     279000|
|   Debit|     132000|
+--------+-----------+



In [53]:
#47
transactions.groupBy("txn_date").count().show()

+----------+-----+
|  txn_date|count|
+----------+-----+
|2024-03-05|    2|
|2024-03-06|    2|
|2024-03-08|    2|
|2024-03-04|    2|
|2024-03-02|    2|
|2024-03-09|    2|
|2024-03-01|    2|
|2024-03-03|    2|
|2024-03-07|    2|
|2024-03-10|    2|
+----------+-----+



In [54]:
#48
customers.groupBy("city").agg(avg("age")).show()

+---------+------------------+
|     city|          avg(age)|
+---------+------------------+
|Bangalore|              30.0|
|  Chennai|              30.0|
|   Mumbai|              34.5|
|     Pune|              38.0|
|    Delhi|28.666666666666668|
|Hyderabad|31.333333333333332|
+---------+------------------+



In [55]:
#49
customers.join(accounts, "customer_id") \
         .groupBy("account_type") \
         .agg(sum("balance")) \
         .show()

+------------+------------+
|account_type|sum(balance)|
+------------+------------+
|     Savings|      645000|
|     Current|      410000|
+------------+------------+



In [56]:
#50
customers.groupBy("city", "account_type").count().show()

+---------+------------+-----+
|     city|account_type|count|
+---------+------------+-----+
|   Mumbai|     Savings|    1|
|Hyderabad|     Savings|    3|
|Bangalore|     Savings|    1|
|Bangalore|     Current|    1|
|   Mumbai|     Current|    1|
|    Delhi|     Savings|    3|
|     Pune|     Current|    1|
|  Chennai|     Current|    1|
+---------+------------+-----+



**Section 6 — Joins**

In [57]:
#51
customers.join(accounts, "customer_id").show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|      1001|   Hyderabad Main|  85000|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      1003|      Mumbai West|  45000|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|
|          8|  Neha|    Delhi|

In [58]:
#52
customers.join(accounts, "customer_id") \
         .select("name", "city", "branch", "balance") \
         .show()

+------+---------+-----------------+-------+
|  name|     city|           branch|balance|
+------+---------+-----------------+-------+
| Rahul|Hyderabad|   Hyderabad Main|  85000|
| Sneha|Bangalore|Bangalore Central| 120000|
| Arjun|   Mumbai|      Mumbai West|  45000|
| Priya|    Delhi|      Delhi North|  95000|
| Karan|  Chennai|    Chennai South|  60000|
| Meera|Hyderabad|   Hyderabad Main| 150000|
|  Amit|     Pune|        Pune East|  30000|
|  Neha|    Delhi|      Delhi North|  70000|
| Divya|Bangalore|Bangalore Central| 110000|
|Vikram|   Mumbai|      Mumbai West| 200000|
|Farhan|Hyderabad|   Hyderabad Main|  50000|
|Simran|    Delhi|      Delhi North|  40000|
+------+---------+-----------------+-------+



In [59]:
#53
accounts.join(transactions, "account_id").show()

+----------+-----------+-----------------+-------+------+--------+------+----------+
|account_id|customer_id|           branch|balance|txn_id|txn_type|amount|  txn_date|
+----------+-----------+-----------------+-------+------+--------+------+----------+
|      1001|          1|   Hyderabad Main|  85000|     1|  Credit| 25000|2024-03-01|
|      1002|          2|Bangalore Central| 120000|     2|   Debit| 15000|2024-03-01|
|      1003|          3|      Mumbai West|  45000|     3|  Credit| 10000|2024-03-02|
|      1004|          4|      Delhi North|  95000|     4|   Debit|  5000|2024-03-02|
|      1005|          5|    Chennai South|  60000|     5|  Credit| 30000|2024-03-03|
|      1006|          6|   Hyderabad Main| 150000|     6|   Debit| 20000|2024-03-03|
|      1007|          7|        Pune East|  30000|     7|  Credit|  8000|2024-03-04|
|      1008|          8|      Delhi North|  70000|     8|   Debit| 12000|2024-03-04|
|      1009|          9|Bangalore Central| 110000|     9|  Credit

In [60]:
#54
accounts.join(transactions, "account_id") \
        .select("account_id", "txn_type", "amount", "balance") \
        .show()

+----------+--------+------+-------+
|account_id|txn_type|amount|balance|
+----------+--------+------+-------+
|      1001|  Credit| 25000|  85000|
|      1002|   Debit| 15000| 120000|
|      1003|  Credit| 10000|  45000|
|      1004|   Debit|  5000|  95000|
|      1005|  Credit| 30000|  60000|
|      1006|   Debit| 20000| 150000|
|      1007|  Credit|  8000|  30000|
|      1008|   Debit| 12000|  70000|
|      1009|  Credit| 40000| 110000|
|      1010|   Debit| 35000| 200000|
|      1001|   Debit|  7000|  85000|
|      1002|  Credit| 18000| 120000|
|      1006|  Credit| 50000| 150000|
|      1010|  Credit| 60000| 200000|
|      1011|   Debit|  9000|  50000|
|      1012|  Credit| 16000|  40000|
|      1003|   Debit|  4000|  45000|
|      1004|  Credit| 22000|  95000|
|      1005|   Debit| 11000|  60000|
|      1009|   Debit| 14000| 110000|
+----------+--------+------+-------+



In [61]:
#55
accounts.join(branches, "branch").show()

+-----------------+----------+-----------+-------+------+-------+
|           branch|account_id|customer_id|balance|region|manager|
+-----------------+----------+-----------+-------+------+-------+
|   Hyderabad Main|      1001|          1|  85000| South| Ramesh|
|Bangalore Central|      1002|          2| 120000| South|  Leena|
|      Mumbai West|      1003|          3|  45000|  West| Joseph|
|      Delhi North|      1004|          4|  95000| North|   Sara|
|    Chennai South|      1005|          5|  60000| South|  Kumar|
|   Hyderabad Main|      1006|          6| 150000| South| Ramesh|
|        Pune East|      1007|          7|  30000|  West|  Anita|
|      Delhi North|      1008|          8|  70000| North|   Sara|
|Bangalore Central|      1009|          9| 110000| South|  Leena|
|      Mumbai West|      1010|         10| 200000|  West| Joseph|
|   Hyderabad Main|      1011|         11|  50000| South| Ramesh|
|      Delhi North|      1012|         12|  40000| North|   Sara|
+---------

In [62]:
#56
accounts.join(branches, "branch") \
        .select("branch", "region", "manager", "balance") \
        .show()

+-----------------+------+-------+-------+
|           branch|region|manager|balance|
+-----------------+------+-------+-------+
|   Hyderabad Main| South| Ramesh|  85000|
|Bangalore Central| South|  Leena| 120000|
|      Mumbai West|  West| Joseph|  45000|
|      Delhi North| North|   Sara|  95000|
|    Chennai South| South|  Kumar|  60000|
|   Hyderabad Main| South| Ramesh| 150000|
|        Pune East|  West|  Anita|  30000|
|      Delhi North| North|   Sara|  70000|
|Bangalore Central| South|  Leena| 110000|
|      Mumbai West|  West| Joseph| 200000|
|   Hyderabad Main| South| Ramesh|  50000|
|      Delhi North| North|   Sara|  40000|
+-----------------+------+-------+-------+



In [63]:
#57
customers.join(accounts, "customer_id") \
         .join(transactions, "account_id") \
         .show()

+----------+-----------+------+---------+---+------------+-----------+-----------------+-------+------+--------+------+----------+
|account_id|customer_id|  name|     city|age|account_type|signup_date|           branch|balance|txn_id|txn_type|amount|  txn_date|
+----------+-----------+------+---------+---+------------+-----------+-----------------+-------+------+--------+------+----------+
|      1001|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|   Hyderabad Main|  85000|    11|   Debit|  7000|2024-03-06|
|      1001|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|   Hyderabad Main|  85000|     1|  Credit| 25000|2024-03-01|
|      1002|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|Bangalore Central| 120000|    12|  Credit| 18000|2024-03-06|
|      1002|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|Bangalore Central| 120000|     2|   Debit| 15000|2024-03-01|
|      1003|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      Mumbai 

In [64]:
#58
customers.join(accounts, "customer_id") \
         .join(transactions, "account_id") \
         .select("name", "city", "txn_type", "amount", "txn_date") \
         .show()

+------+---------+--------+------+----------+
|  name|     city|txn_type|amount|  txn_date|
+------+---------+--------+------+----------+
| Rahul|Hyderabad|   Debit|  7000|2024-03-06|
| Rahul|Hyderabad|  Credit| 25000|2024-03-01|
| Sneha|Bangalore|  Credit| 18000|2024-03-06|
| Sneha|Bangalore|   Debit| 15000|2024-03-01|
| Arjun|   Mumbai|   Debit|  4000|2024-03-09|
| Arjun|   Mumbai|  Credit| 10000|2024-03-02|
| Priya|    Delhi|  Credit| 22000|2024-03-09|
| Priya|    Delhi|   Debit|  5000|2024-03-02|
| Karan|  Chennai|   Debit| 11000|2024-03-10|
| Karan|  Chennai|  Credit| 30000|2024-03-03|
| Meera|Hyderabad|  Credit| 50000|2024-03-07|
| Meera|Hyderabad|   Debit| 20000|2024-03-03|
|  Amit|     Pune|  Credit|  8000|2024-03-04|
|  Neha|    Delhi|   Debit| 12000|2024-03-04|
| Divya|Bangalore|   Debit| 14000|2024-03-10|
| Divya|Bangalore|  Credit| 40000|2024-03-05|
|Vikram|   Mumbai|  Credit| 60000|2024-03-07|
|Vikram|   Mumbai|   Debit| 35000|2024-03-05|
|Farhan|Hyderabad|   Debit|  9000|

In [65]:
#59
customers.join(accounts, "customer_id") \
         .join(branches, "branch") \
         .join(transactions, "account_id") \
         .show()

+----------+-----------------+-----------+------+---------+---+------------+-----------+-------+------+-------+------+--------+------+----------+
|account_id|           branch|customer_id|  name|     city|age|account_type|signup_date|balance|region|manager|txn_id|txn_type|amount|  txn_date|
+----------+-----------------+-----------+------+---------+---+------------+-----------+-------+------+-------+------+--------+------+----------+
|      1001|   Hyderabad Main|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  85000| South| Ramesh|    11|   Debit|  7000|2024-03-06|
|      1001|   Hyderabad Main|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  85000| South| Ramesh|     1|  Credit| 25000|2024-03-01|
|      1002|Bangalore Central|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| 120000| South|  Leena|    12|  Credit| 18000|2024-03-06|
|      1002|Bangalore Central|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| 120000| South|  Leena|     2|   Deb

In [66]:
#60
customers.join(accounts, "customer_id") \
         .join(transactions, "account_id") \
         .groupBy("name") \
         .agg(sum("amount")) \
         .show()

+------+-----------+
|  name|sum(amount)|
+------+-----------+
| Divya|      54000|
| Meera|      70000|
| Sneha|      33000|
| Priya|      27000|
|Vikram|      95000|
|Simran|      16000|
| Rahul|      32000|
| Arjun|      14000|
|  Amit|       8000|
|  Neha|      12000|
|Farhan|       9000|
| Karan|      41000|
+------+-----------+



**Section 7 — withColumn**

In [67]:
#61
accounts.withColumn("balance_in_lakhs", col("balance") / 100000).show()

+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_in_lakhs|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|            0.85|
|      1002|          2|Bangalore Central| 120000|             1.2|
|      1003|          3|      Mumbai West|  45000|            0.45|
|      1004|          4|      Delhi North|  95000|            0.95|
|      1005|          5|    Chennai South|  60000|             0.6|
|      1006|          6|   Hyderabad Main| 150000|             1.5|
|      1007|          7|        Pune East|  30000|             0.3|
|      1008|          8|      Delhi North|  70000|             0.7|
|      1009|          9|Bangalore Central| 110000|             1.1|
|      1010|         10|      Mumbai West| 200000|             2.0|
|      1011|         11|   Hyderabad Main|  50000|             0.5|
|      1012|         12|      Delhi North|  4000

In [68]:
#62
customers.withColumn("bank_name", lit("BotCampus Bank")).show()

+-----------+------+---------+---+------------+-----------+--------------+
|customer_id|  name|     city|age|account_type|signup_date|     bank_name|
+-----------+------+---------+---+------------+-----------+--------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|BotCampus Bank|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|BotCampus Bank|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|BotCampus Bank|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|BotCampus Bank|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|BotCampus Bank|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|BotCampus Bank|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|BotCampus Bank|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|BotCampus Bank|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|BotCampus Bank|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|BotCampus Bank|
|         11|Farhan|Hyder

In [69]:
#63
accounts.withColumn("annual_service_fee", col("balance") * 0.01).show()

+----------+-----------+-----------------+-------+------------------+
|account_id|customer_id|           branch|balance|annual_service_fee|
+----------+-----------+-----------------+-------+------------------+
|      1001|          1|   Hyderabad Main|  85000|             850.0|
|      1002|          2|Bangalore Central| 120000|            1200.0|
|      1003|          3|      Mumbai West|  45000|             450.0|
|      1004|          4|      Delhi North|  95000|             950.0|
|      1005|          5|    Chennai South|  60000|             600.0|
|      1006|          6|   Hyderabad Main| 150000|            1500.0|
|      1007|          7|        Pune East|  30000|             300.0|
|      1008|          8|      Delhi North|  70000|             700.0|
|      1009|          9|Bangalore Central| 110000|            1100.0|
|      1010|         10|      Mumbai West| 200000|            2000.0|
|      1011|         11|   Hyderabad Main|  50000|             500.0|
|      1012|        

In [70]:
#64
accounts.withColumn("annual_service_fee", col("balance") * 0.01) \
        .withColumn("net_balance", col("balance") - col("annual_service_fee")) \
        .show()

+----------+-----------+-----------------+-------+------------------+-----------+
|account_id|customer_id|           branch|balance|annual_service_fee|net_balance|
+----------+-----------+-----------------+-------+------------------+-----------+
|      1001|          1|   Hyderabad Main|  85000|             850.0|    84150.0|
|      1002|          2|Bangalore Central| 120000|            1200.0|   118800.0|
|      1003|          3|      Mumbai West|  45000|             450.0|    44550.0|
|      1004|          4|      Delhi North|  95000|             950.0|    94050.0|
|      1005|          5|    Chennai South|  60000|             600.0|    59400.0|
|      1006|          6|   Hyderabad Main| 150000|            1500.0|   148500.0|
|      1007|          7|        Pune East|  30000|             300.0|    29700.0|
|      1008|          8|      Delhi North|  70000|             700.0|    69300.0|
|      1009|          9|Bangalore Central| 110000|            1100.0|   108900.0|
|      1010|    

In [71]:
#65
accounts.withColumn("is_high_balance",
    when(col("balance") > 100000, "Yes").otherwise("No")
).show()

+----------+-----------+-----------------+-------+---------------+
|account_id|customer_id|           branch|balance|is_high_balance|
+----------+-----------+-----------------+-------+---------------+
|      1001|          1|   Hyderabad Main|  85000|             No|
|      1002|          2|Bangalore Central| 120000|            Yes|
|      1003|          3|      Mumbai West|  45000|             No|
|      1004|          4|      Delhi North|  95000|             No|
|      1005|          5|    Chennai South|  60000|             No|
|      1006|          6|   Hyderabad Main| 150000|            Yes|
|      1007|          7|        Pune East|  30000|             No|
|      1008|          8|      Delhi North|  70000|             No|
|      1009|          9|Bangalore Central| 110000|            Yes|
|      1010|         10|      Mumbai West| 200000|            Yes|
|      1011|         11|   Hyderabad Main|  50000|             No|
|      1012|         12|      Delhi North|  40000|            

In [72]:
#66
transactions.withColumn("txn_amount_in_k", col("amount") / 1000).show()

+------+----------+--------+------+----------+---------------+
|txn_id|account_id|txn_type|amount|  txn_date|txn_amount_in_k|
+------+----------+--------+------+----------+---------------+
|     1|      1001|  Credit| 25000|2024-03-01|           25.0|
|     2|      1002|   Debit| 15000|2024-03-01|           15.0|
|     3|      1003|  Credit| 10000|2024-03-02|           10.0|
|     4|      1004|   Debit|  5000|2024-03-02|            5.0|
|     5|      1005|  Credit| 30000|2024-03-03|           30.0|
|     6|      1006|   Debit| 20000|2024-03-03|           20.0|
|     7|      1007|  Credit|  8000|2024-03-04|            8.0|
|     8|      1008|   Debit| 12000|2024-03-04|           12.0|
|     9|      1009|  Credit| 40000|2024-03-05|           40.0|
|    10|      1010|   Debit| 35000|2024-03-05|           35.0|
|    11|      1001|   Debit|  7000|2024-03-06|            7.0|
|    12|      1002|  Credit| 18000|2024-03-06|           18.0|
|    13|      1006|  Credit| 50000|2024-03-07|         

In [73]:
#67
customers.withColumn("country", lit("India")).show()

+-----------+------+---------+---+------------+-----------+-------+
|customer_id|  name|     city|age|account_type|signup_date|country|
+-----------+------+---------+---+------------+-----------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  India|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|  India|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|  India|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|  India|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|  India|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|  India|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|  India|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|  India|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|  India|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|  India|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|  India|
|         12|Simran|    Delhi| 25|     Savings| 

In [74]:
#68
customers.withColumn("customer_label",
    concat(col("name"), lit(" - "), col("city"))
).show()

+-----------+------+---------+---+------------+-----------+------------------+
|customer_id|  name|     city|age|account_type|signup_date|    customer_label|
+-----------+------+---------+---+------------+-----------+------------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10| Rahul - Hyderabad|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| Sneha - Bangalore|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Arjun - Mumbai|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|     Priya - Delhi|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|   Karan - Chennai|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10| Meera - Hyderabad|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|       Amit - Pune|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      Neha - Delhi|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15| Divya - Bangalore|
|         10|Vikram|   Mumbai| 42|     Current| 2023

In [75]:
#69
branches.withColumn("branch_label",
    concat(col("branch"), lit(" - "), col("region"))
).show()

+-----------------+------+-------+--------------------+
|           branch|region|manager|        branch_label|
+-----------------+------+-------+--------------------+
|   Hyderabad Main| South| Ramesh|Hyderabad Main - ...|
|Bangalore Central| South|  Leena|Bangalore Central...|
|      Mumbai West|  West| Joseph|  Mumbai West - West|
|      Delhi North| North|   Sara| Delhi North - North|
|    Chennai South| South|  Kumar|Chennai South - S...|
|        Pune East|  West|  Anita|    Pune East - West|
+-----------------+------+-------+--------------------+



In [76]:
#70
transactions.withColumn("risk_flag",
    when(col("amount") > 40000, "High Risk").otherwise("Normal")
).show()

+------+----------+--------+------+----------+---------+
|txn_id|account_id|txn_type|amount|  txn_date|risk_flag|
+------+----------+--------+------+----------+---------+
|     1|      1001|  Credit| 25000|2024-03-01|   Normal|
|     2|      1002|   Debit| 15000|2024-03-01|   Normal|
|     3|      1003|  Credit| 10000|2024-03-02|   Normal|
|     4|      1004|   Debit|  5000|2024-03-02|   Normal|
|     5|      1005|  Credit| 30000|2024-03-03|   Normal|
|     6|      1006|   Debit| 20000|2024-03-03|   Normal|
|     7|      1007|  Credit|  8000|2024-03-04|   Normal|
|     8|      1008|   Debit| 12000|2024-03-04|   Normal|
|     9|      1009|  Credit| 40000|2024-03-05|   Normal|
|    10|      1010|   Debit| 35000|2024-03-05|   Normal|
|    11|      1001|   Debit|  7000|2024-03-06|   Normal|
|    12|      1002|  Credit| 18000|2024-03-06|   Normal|
|    13|      1006|  Credit| 50000|2024-03-07|High Risk|
|    14|      1010|  Credit| 60000|2024-03-07|High Risk|
|    15|      1011|   Debit|  9

**Section 8 — when / otherwise**

In [77]:
#71
accounts.withColumn("balance_category",
    when(col("balance") >= 100000, "High")
    .when(col("balance") >= 50000, "Medium")
    .otherwise("Low")
).show()

+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_category|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|          Medium|
|      1002|          2|Bangalore Central| 120000|            High|
|      1003|          3|      Mumbai West|  45000|             Low|
|      1004|          4|      Delhi North|  95000|          Medium|
|      1005|          5|    Chennai South|  60000|          Medium|
|      1006|          6|   Hyderabad Main| 150000|            High|
|      1007|          7|        Pune East|  30000|             Low|
|      1008|          8|      Delhi North|  70000|          Medium|
|      1009|          9|Bangalore Central| 110000|            High|
|      1010|         10|      Mumbai West| 200000|            High|
|      1011|         11|   Hyderabad Main|  50000|          Medium|
|      1012|         12|      Delhi North|  4000

In [78]:
#72
customers.withColumn("age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 40, "Adult")
    .otherwise("Senior")
).show()

+-----------+------+---------+---+------------+-----------+---------+
|customer_id|  name|     city|age|account_type|signup_date|age_group|
+-----------+------+---------+---+------------+-----------+---------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|    Young|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|    Adult|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Young|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|    Adult|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|    Adult|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|    Adult|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|    Adult|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|    Young|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|    Young|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|   Senior|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|    Adult|
|         12|Simran|

In [79]:
#73
transactions.withColumn("transaction_category",
    when(col("amount") >= 30000, "Large")
    .when(col("amount") >= 10000, "Medium")
    .otherwise("Small")
).show()

+------+----------+--------+------+----------+--------------------+
|txn_id|account_id|txn_type|amount|  txn_date|transaction_category|
+------+----------+--------+------+----------+--------------------+
|     1|      1001|  Credit| 25000|2024-03-01|              Medium|
|     2|      1002|   Debit| 15000|2024-03-01|              Medium|
|     3|      1003|  Credit| 10000|2024-03-02|              Medium|
|     4|      1004|   Debit|  5000|2024-03-02|               Small|
|     5|      1005|  Credit| 30000|2024-03-03|               Large|
|     6|      1006|   Debit| 20000|2024-03-03|              Medium|
|     7|      1007|  Credit|  8000|2024-03-04|               Small|
|     8|      1008|   Debit| 12000|2024-03-04|              Medium|
|     9|      1009|  Credit| 40000|2024-03-05|               Large|
|    10|      1010|   Debit| 35000|2024-03-05|               Large|
|    11|      1001|   Debit|  7000|2024-03-06|               Small|
|    12|      1002|  Credit| 18000|2024-03-06|  

In [80]:
#74
branches.withColumn("region_priority",
    when(col("region") == "South", "High Priority")
    .when(col("region") == "North", "Medium Priority")
    .otherwise("Watch")
).show()

+-----------------+------+-------+---------------+
|           branch|region|manager|region_priority|
+-----------------+------+-------+---------------+
|   Hyderabad Main| South| Ramesh|  High Priority|
|Bangalore Central| South|  Leena|  High Priority|
|      Mumbai West|  West| Joseph|          Watch|
|      Delhi North| North|   Sara|Medium Priority|
|    Chennai South| South|  Kumar|  High Priority|
|        Pune East|  West|  Anita|          Watch|
+-----------------+------+-------+---------------+



In [81]:
#75
customers.withColumn("account_type_label",
    when(col("account_type") == "Savings", "Retail")
    .otherwise("Business")
).show()

+-----------+------+---------+---+------------+-----------+------------------+
|customer_id|  name|     city|age|account_type|signup_date|account_type_label|
+-----------+------+---------+---+------------+-----------+------------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|            Retail|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|          Business|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|            Retail|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|            Retail|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|          Business|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|            Retail|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|          Business|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|            Retail|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|            Retail|
|         10|Vikram|   Mumbai| 42|     Current| 2023

**Section 9 — Date Functions**

In [82]:
#76
customers.withColumn("signup_date", to_date("signup_date")).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [83]:
#77
customers.withColumn("signup_year", year("signup_date")).show()

+-----------+------+---------+---+------------+-----------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|signup_year|
+-----------+------+---------+---+------------+-----------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|       2023|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|       2023|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|       2023|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|       2023|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|       2023|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|       2023|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|       2023|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|       2023|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|       2023|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|       2023|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|     

In [84]:
#78
customers.withColumn("signup_month", month("signup_date")).show()

+-----------+------+---------+---+------------+-----------+------------+
|customer_id|  name|     city|age|account_type|signup_date|signup_month|
+-----------+------+---------+---+------------+-----------+------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|           1|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|           2|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|           3|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|           4|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|           5|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|           6|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|           6|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|           7|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|           7|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|           8|
|         11|Farhan|Hyderabad| 34|     Savings| 202

In [85]:
#79
transactions.withColumn("txn_date", to_date("txn_date")).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|    11|      1001|   Debit|  7000|2024-03-06|
|    12|      1002|  Credit| 18000|2024-03-06|
|    13|      1006|  Credit| 50000|2024-03-07|
|    14|      1010|  Credit| 60000|2024-03-07|
|    15|      1011|   Debit|  9000|2024-03-08|
|    16|      1012|  Credit| 16000|2024-03-08|
|    17|      1003|   Debit|  4000|2024-03-09|
|    18|      1004|  Credit| 22000|2024-03-09|
|    19|     

In [86]:
#80
transactions.withColumn("txn_month", month("txn_date")).show()

+------+----------+--------+------+----------+---------+
|txn_id|account_id|txn_type|amount|  txn_date|txn_month|
+------+----------+--------+------+----------+---------+
|     1|      1001|  Credit| 25000|2024-03-01|        3|
|     2|      1002|   Debit| 15000|2024-03-01|        3|
|     3|      1003|  Credit| 10000|2024-03-02|        3|
|     4|      1004|   Debit|  5000|2024-03-02|        3|
|     5|      1005|  Credit| 30000|2024-03-03|        3|
|     6|      1006|   Debit| 20000|2024-03-03|        3|
|     7|      1007|  Credit|  8000|2024-03-04|        3|
|     8|      1008|   Debit| 12000|2024-03-04|        3|
|     9|      1009|  Credit| 40000|2024-03-05|        3|
|    10|      1010|   Debit| 35000|2024-03-05|        3|
|    11|      1001|   Debit|  7000|2024-03-06|        3|
|    12|      1002|  Credit| 18000|2024-03-06|        3|
|    13|      1006|  Credit| 50000|2024-03-07|        3|
|    14|      1010|  Credit| 60000|2024-03-07|        3|
|    15|      1011|   Debit|  9

In [87]:
#81
transactions.groupBy("txn_date").count().show()

+----------+-----+
|  txn_date|count|
+----------+-----+
|2024-03-05|    2|
|2024-03-06|    2|
|2024-03-08|    2|
|2024-03-04|    2|
|2024-03-02|    2|
|2024-03-09|    2|
|2024-03-01|    2|
|2024-03-03|    2|
|2024-03-07|    2|
|2024-03-10|    2|
+----------+-----+



In [88]:
#82
transactions.withColumn("txn_month", month("txn_date")) \
            .groupBy("txn_month") \
            .count() \
            .show()

+---------+-----+
|txn_month|count|
+---------+-----+
|        3|   20|
+---------+-----+



In [89]:
#83
customers.filter(col("signup_date") > "2023-06-01").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [90]:
#84
customers.withColumn("days_since_signup",
    datediff(current_date(), col("signup_date"))
).show()

+-----------+------+---------+---+------------+-----------+-----------------+
|customer_id|  name|     city|age|account_type|signup_date|days_since_signup|
+-----------+------+---------+---+------------+-----------+-----------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|             1203|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|             1170|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|             1140|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|             1108|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|             1082|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|             1052|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|             1040|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|             1022|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|             1017|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|     

In [91]:
#85
transactions.withColumn("days_since_transaction",
    datediff(current_date(), col("txn_date"))
).show()

+------+----------+--------+------+----------+----------------------+
|txn_id|account_id|txn_type|amount|  txn_date|days_since_transaction|
+------+----------+--------+------+----------+----------------------+
|     1|      1001|  Credit| 25000|2024-03-01|                   787|
|     2|      1002|   Debit| 15000|2024-03-01|                   787|
|     3|      1003|  Credit| 10000|2024-03-02|                   786|
|     4|      1004|   Debit|  5000|2024-03-02|                   786|
|     5|      1005|  Credit| 30000|2024-03-03|                   785|
|     6|      1006|   Debit| 20000|2024-03-03|                   785|
|     7|      1007|  Credit|  8000|2024-03-04|                   784|
|     8|      1008|   Debit| 12000|2024-03-04|                   784|
|     9|      1009|  Credit| 40000|2024-03-05|                   783|
|    10|      1010|   Debit| 35000|2024-03-05|                   783|
|    11|      1001|   Debit|  7000|2024-03-06|                   782|
|    12|      1002| 

**Section 10 — Window Functions**

In [92]:
#86
cust_bal = customers.join(accounts, "customer_id")
w1 = Window.partitionBy("city").orderBy(col("balance").desc())

cust_bal.withColumn("rank", rank().over(w1)).show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|rank|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|   1|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|      1009|Bangalore Central| 110000|   2|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|   1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|   1|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      1008|      Delhi North|  70000|   2|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|      1012|      Delhi North|  40000|   3|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad

In [93]:
#87
cust_bal.withColumn("row_num", row_number().over(w1)) \
        .filter(col("row_num") == 1) \
        .show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|row_num|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+-------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|      1|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|      1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|      1|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|      1|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|      1010|      Mumbai West| 200000|      1|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|      1|
+-----------+------+---------+---+------------+--------

In [95]:
#88
w2 = Window.partitionBy("txn_type").orderBy(col("amount").desc())
transactions.withColumn("dense_rank", dense_rank().over(w2)).show()

+------+----------+--------+------+----------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|dense_rank|
+------+----------+--------+------+----------+----------+
|    14|      1010|  Credit| 60000|2024-03-07|         1|
|    13|      1006|  Credit| 50000|2024-03-07|         2|
|     9|      1009|  Credit| 40000|2024-03-05|         3|
|     5|      1005|  Credit| 30000|2024-03-03|         4|
|     1|      1001|  Credit| 25000|2024-03-01|         5|
|    18|      1004|  Credit| 22000|2024-03-09|         6|
|    12|      1002|  Credit| 18000|2024-03-06|         7|
|    16|      1012|  Credit| 16000|2024-03-08|         8|
|     3|      1003|  Credit| 10000|2024-03-02|         9|
|     7|      1007|  Credit|  8000|2024-03-04|        10|
|    10|      1010|   Debit| 35000|2024-03-05|         1|
|     6|      1006|   Debit| 20000|2024-03-03|         2|
|     2|      1002|   Debit| 15000|2024-03-01|         3|
|    20|      1009|   Debit| 14000|2024-03-10|         4|
|     8|      

In [96]:
#89
w3 = Window.partitionBy("account_id").orderBy(col("amount").desc())

transactions.withColumn("rank", rank().over(w3)) \
            .filter(col("rank") <= 2) \
            .show()

+------+----------+--------+------+----------+----+
|txn_id|account_id|txn_type|amount|  txn_date|rank|
+------+----------+--------+------+----------+----+
|     1|      1001|  Credit| 25000|2024-03-01|   1|
|    11|      1001|   Debit|  7000|2024-03-06|   2|
|    12|      1002|  Credit| 18000|2024-03-06|   1|
|     2|      1002|   Debit| 15000|2024-03-01|   2|
|     3|      1003|  Credit| 10000|2024-03-02|   1|
|    17|      1003|   Debit|  4000|2024-03-09|   2|
|    18|      1004|  Credit| 22000|2024-03-09|   1|
|     4|      1004|   Debit|  5000|2024-03-02|   2|
|     5|      1005|  Credit| 30000|2024-03-03|   1|
|    19|      1005|   Debit| 11000|2024-03-10|   2|
|    13|      1006|  Credit| 50000|2024-03-07|   1|
|     6|      1006|   Debit| 20000|2024-03-03|   2|
|     7|      1007|  Credit|  8000|2024-03-04|   1|
|     8|      1008|   Debit| 12000|2024-03-04|   1|
|     9|      1009|  Credit| 40000|2024-03-05|   1|
|    20|      1009|   Debit| 14000|2024-03-10|   2|
|    14|    

In [97]:
#90
w4 = Window.partitionBy("account_id").orderBy("txn_date")

transactions.withColumn("running_total", sum("amount").over(w4)).show()

+------+----------+--------+------+----------+-------------+
|txn_id|account_id|txn_type|amount|  txn_date|running_total|
+------+----------+--------+------+----------+-------------+
|     1|      1001|  Credit| 25000|2024-03-01|        25000|
|    11|      1001|   Debit|  7000|2024-03-06|        32000|
|     2|      1002|   Debit| 15000|2024-03-01|        15000|
|    12|      1002|  Credit| 18000|2024-03-06|        33000|
|     3|      1003|  Credit| 10000|2024-03-02|        10000|
|    17|      1003|   Debit|  4000|2024-03-09|        14000|
|     4|      1004|   Debit|  5000|2024-03-02|         5000|
|    18|      1004|  Credit| 22000|2024-03-09|        27000|
|     5|      1005|  Credit| 30000|2024-03-03|        30000|
|    19|      1005|   Debit| 11000|2024-03-10|        41000|
|     6|      1006|   Debit| 20000|2024-03-03|        20000|
|    13|      1006|  Credit| 50000|2024-03-07|        70000|
|     7|      1007|  Credit|  8000|2024-03-04|         8000|
|     8|      1008|   De

In [98]:
#91
branch_balance = accounts.groupBy("branch").agg(sum("balance").alias("total_balance"))

branch_balance.withColumn("rank",
    rank().over(Window.orderBy(col("total_balance").desc()))
).show()

+-----------------+-------------+----+
|           branch|total_balance|rank|
+-----------------+-------------+----+
|   Hyderabad Main|       285000|   1|
|      Mumbai West|       245000|   2|
|Bangalore Central|       230000|   3|
|      Delhi North|       205000|   4|
|    Chennai South|        60000|   5|
|        Pune East|        30000|   6|
+-----------------+-------------+----+



In [99]:
#92
city_balance = customers.join(accounts, "customer_id") \
                        .groupBy("city") \
                        .agg(sum("balance").alias("total_balance"))

city_balance.withColumn("rank",
    rank().over(Window.orderBy(col("total_balance").desc()))
).show()

+---------+-------------+----+
|     city|total_balance|rank|
+---------+-------------+----+
|Hyderabad|       285000|   1|
|   Mumbai|       245000|   2|
|Bangalore|       230000|   3|
|    Delhi|       205000|   4|
|  Chennai|        60000|   5|
|     Pune|        30000|   6|
+---------+-------------+----+



In [100]:
#93
customer_txn = customers.join(accounts, "customer_id") \
                        .join(transactions, "account_id") \
                        .groupBy("name") \
                        .agg(sum("amount").alias("total_txn_amount"))

customer_txn.withColumn("rank",
    rank().over(Window.orderBy(col("total_txn_amount").desc()))
).show()

+------+----------------+----+
|  name|total_txn_amount|rank|
+------+----------------+----+
|Vikram|           95000|   1|
| Meera|           70000|   2|
| Divya|           54000|   3|
| Karan|           41000|   4|
| Sneha|           33000|   5|
| Rahul|           32000|   6|
| Priya|           27000|   7|
|Simran|           16000|   8|
| Arjun|           14000|   9|
|  Neha|           12000|  10|
|Farhan|            9000|  11|
|  Amit|            8000|  12|
+------+----------------+----+



In [101]:
#94
city_txn = customers.join(accounts, "customer_id") \
                    .join(transactions, "account_id")

w5 = Window.partitionBy("city").orderBy(col("amount").desc())

city_txn.withColumn("rank", rank().over(w5)) \
        .filter(col("rank") == 1) \
        .select("city", "name", "amount", "txn_type") \
        .show()

+---------+------+------+--------+
|     city|  name|amount|txn_type|
+---------+------+------+--------+
|Bangalore| Divya| 40000|  Credit|
|  Chennai| Karan| 30000|  Credit|
|    Delhi| Priya| 22000|  Credit|
|Hyderabad| Meera| 50000|  Credit|
|   Mumbai|Vikram| 60000|  Credit|
|     Pune|  Amit|  8000|  Credit|
+---------+------+------+--------+



In [102]:
#95
region_bal = customers.join(accounts, "customer_id") \
                      .join(branches, "branch")

w6 = Window.partitionBy("region").orderBy(col("balance").desc())

region_bal.withColumn("rank", rank().over(w6)) \
          .filter(col("rank") == 1) \
          .select("region", "name", "branch", "balance") \
          .show()

+------+------+--------------+-------+
|region|  name|        branch|balance|
+------+------+--------------+-------+
| North| Priya|   Delhi North|  95000|
| South| Meera|Hyderabad Main| 150000|
|  West|Vikram|   Mumbai West| 200000|
+------+------+--------------+-------+



**Section 11 — Complex JSON**

In [103]:
#96
profiles = spark.read.json("customer_profiles.json", multiLine=True)
profiles.show()

+--------------------+-----------+------+--------------------+
|             contact|customer_id|  name|            services|
+--------------------+-----------+------+--------------------+
|{rahul@mail.com, ...|          1| Rahul|[UPI, Credit Card...|
|{sneha@mail.com, ...|          2| Sneha|   [UPI, Debit Card]|
|{arjun@mail.com, ...|          3| Arjun| [Net Banking, Loan]|
|{meera@mail.com, ...|          6| Meera|[UPI, Credit Card...|
|{vikram@mail.com,...|         10|Vikram|[Net Banking, Wea...|
+--------------------+-----------+------+--------------------+



In [104]:
#97
profiles.select(
    "customer_id",
    "name",
    col("contact.email").alias("email"),
    col("contact.phone").alias("phone")
).show()

+-----------+------+---------------+----------+
|customer_id|  name|          email|     phone|
+-----------+------+---------------+----------+
|          1| Rahul| rahul@mail.com|9000011111|
|          2| Sneha| sneha@mail.com|9000022222|
|          3| Arjun| arjun@mail.com|9000033333|
|          6| Meera| meera@mail.com|9000066666|
|         10|Vikram|vikram@mail.com|9000101010|
+-----------+------+---------------+----------+



In [105]:
#98
profiles.select(
    "customer_id",
    "name",
    explode("services").alias("service")
).show()

+-----------+------+-----------+
|customer_id|  name|    service|
+-----------+------+-----------+
|          1| Rahul|        UPI|
|          1| Rahul|Credit Card|
|          1| Rahul|Net Banking|
|          2| Sneha|        UPI|
|          2| Sneha| Debit Card|
|          3| Arjun|Net Banking|
|          3| Arjun|       Loan|
|          6| Meera|        UPI|
|          6| Meera|Credit Card|
|          6| Meera|       Loan|
|         10|Vikram|Net Banking|
|         10|Vikram|     Wealth|
+-----------+------+-----------+



In [106]:
#99
profiles.filter(array_contains(col("services"), "UPI")).show()

+--------------------+-----------+-----+--------------------+
|             contact|customer_id| name|            services|
+--------------------+-----------+-----+--------------------+
|{rahul@mail.com, ...|          1|Rahul|[UPI, Credit Card...|
|{sneha@mail.com, ...|          2|Sneha|   [UPI, Debit Card]|
|{meera@mail.com, ...|          6|Meera|[UPI, Credit Card...|
+--------------------+-----------+-----+--------------------+



In [107]:
#100
profiles.select(explode("services").alias("service")) \
        .groupBy("service") \
        .count() \
        .show()

+-----------+-----+
|    service|count|
+-----------+-----+
|     Wealth|    1|
|       Loan|    2|
|Credit Card|    2|
|Net Banking|    3|
| Debit Card|    1|
|        UPI|    3|
+-----------+-----+

